### Step 1: Load Libraries

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pprint import pprint

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing!")
else:
    print("Key is: " + OPENAI_API_KEY[:8])

Key is: sk-proj-


### Step 2: Setup PushOver

In [8]:
load_dotenv()

pushover_user  = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url   = "https://api.pushover.net/1/messages.json"

print(pushover_user)
print(pushover_token)

Ut5xparzfvrod1rjygf36f7h1vqmx2
ahbf3s9qno83otx5dtgpr1bhknawj1


In [9]:
#Test Pushover
import requests

def send_notification(message: str):
    pay_load = {"user":pushover_user,
                "token": pushover_token,
                "message": message}
    requests.post(pushover_url, data = pay_load)

send_notification("Hello to Si Lam")
    

### Step 3: Describe Pushover in LLM

In [20]:
send_notification_function = {

    "name" : "send_notification",
    "description": "Send a push notification to user phone via pushover. Tell the user important task, event",
    "parameters": {
        "type": "object",
        "properties":{
            "message": {
                "type": "string",
                "description":"The notification message to send to user device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pushover to the list of tools for LLM

In [21]:
tools = [
    {"type": "function","function": send_notification_function}
]

### Step 5: calling the tool from LLM

In [22]:
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role":"user", "content":"Please send me a notification telling me what amzing progress I am making on the AI Engineering trainging by SuperDataScience"}
    ],
    tools= tools,
    tool_choice="auto"
)

#check if the model wants to call a tool

message = response.choices[0].message
pprint(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_QHcYeQYa8Gq4GCppKuscvLIt', function=Function(arguments='{"message":"You are making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work!"}', name='send_notification'), type='function')])


In [23]:
if message.tool_calls:
    tool_call = message.tool_calls[0]
    import json
    args = json.loads(tool_call.function.arguments)

    #Actually send notification

    send_notification(args["message"])

    print(f"Sent notification {args['message']}")

else:
    print(message.content)


Sent notification You are making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work!
